# Notebook 6: Corpus Analysis and Visualizations

Produces the knowledge graph visualizations (Figures 2 and 3).

**Analyses:**
- Inter-document semantic similarity (Figure 2)
- Retrieval quality by document (Figure 3)
- Correct document retrieval rate
- Corpus homogeneity analysis

**Runtime:** ~10 min

In [ ]:
!pip install -q sentence-transformers faiss-cpu keybert pyvis networkx

import pickle, json, numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
from pyvis.network import Network
import faiss

In [ ]:
# Load artifacts
with open('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/chunks.pkl', 'rb') as f:
    all_chunks = pickle.load(f)
index = faiss.read_index('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/faiss_index.bin')

# Load extraction results
log_df = pd.read_csv('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/extraction_results.csv')
rqa_df = pd.read_csv('/kaggle/input/regulationqa-artifacts/regulationqa_artifacts/regulationqa_v1.csv')

# Load BGE for centroids
bge_model  = SentenceTransformer('BAAI/bge-base-en-v1.5')
embeddings = bge_model.encode([c['text'] for c in all_chunks],
                               batch_size=64, show_progress_bar=True,
                               normalize_embeddings=True)
print(f'Embeddings: {embeddings.shape}')

In [ ]:
# Compute document centroids
from collections import defaultdict
doc_chunk_map = defaultdict(list)
for i, c in enumerate(all_chunks):
    doc_chunk_map[c['doc_name']].append(i)

doc_centroids = {}
for doc_name, chunk_indices in doc_chunk_map.items():
    chunk_embs = embeddings[chunk_indices]
    centroid   = chunk_embs.mean(axis=0)
    centroid   = centroid / np.linalg.norm(centroid)
    doc_centroids[doc_name] = centroid

print(f'Computed centroids for {len(doc_centroids)} documents.')

In [ ]:
# Inter-document similarity
doc_names = list(doc_centroids.keys())
n = len(doc_names)
sim_matrix = np.zeros((n, n))
for i, d1 in enumerate(doc_names):
    for j, d2 in enumerate(doc_names):
        sim_matrix[i][j] = np.dot(doc_centroids[d1], doc_centroids[d2])

# Most similar pairs
pairs = []
for i in range(n):
    for j in range(i+1, n):
        pairs.append((sim_matrix[i][j], doc_names[i], doc_names[j]))
pairs.sort(reverse=True)

print('Top 15 most similar document pairs:')
print(f'{"Similarity":>12}  Document 1 vs Document 2')
print('-' * 80)
for sim, d1, d2 in pairs[:15]:
    print(f'{sim:>12.4f}  {d1[:35]} ↔ {d2[:35]}')

# Save matrix
sim_df = pd.DataFrame(sim_matrix, index=doc_names, columns=doc_names)
sim_df.to_csv('/kaggle/working/doc_similarity_matrix.csv')
print('\nSimilarity matrix saved.')

In [ ]:
# Document retrieval quality stats
print('\nRetrieval quality by document source:')
doc_f1 = log_df.groupby('doc_source')['f1'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print(doc_f1.to_string())

In [ ]:
# Figure 2: Document Similarity Knowledge Graph
jurisdiction_colors = {
    'Indian statute':    '#FF6B6B',
    'Indian rules':      '#FF8E53',
    'Indian direction':  '#FFA500',
    'Indian guideline':  '#FFD93D',
    'European statute':  '#6BCB77',
    'Intl. standard':    '#4D96FF',
    'Indian policy':     '#FF6B9D',
}

doc_meta = {
    'DPDP Act 2023':                        ('Indian statute',   len(doc_chunk_map.get('DPDP Act 2023', []))),
    'Digital Personal Data Protection Rules 2025': ('Indian rules', len(doc_chunk_map.get('Digital Personal Data Protection Rules 2025', []))),
    'CERT-In Directions 2022':              ('Indian direction', len(doc_chunk_map.get('CERT-In Directions 2022', []))),
    'RBI Cyber Security Framework':         ('Indian direction', len(doc_chunk_map.get('RBI Cyber Security Framework', []))),
    'SEBI CSCRF':                           ('Indian direction', len(doc_chunk_map.get('SEBI CSCRF', []))),
    'State Data Centre security guidelines': ('Indian guideline', len(doc_chunk_map.get('State Data Centre security guidelines', []))),
    'Cyber Security Power Sector 2021':     ('Indian guideline', len(doc_chunk_map.get('Cyber Security Power Sector 2021', []))),
    'NCIIPC Guidelines':                    ('Indian guideline', len(doc_chunk_map.get('NCIIPC Guidelines', []))),
    'IT Protected System Rules 2018':       ('Indian rules',    len(doc_chunk_map.get('IT Protected System Rules 2018', []))),
    'IT Act 2000':                          ('Indian statute',  len(doc_chunk_map.get('IT Act 2000', []))),
    'Electricity Act 2003':                 ('Indian statute',  len(doc_chunk_map.get('Electricity Act 2003', []))),
    'e-Governance Policy India':            ('Indian policy',   len(doc_chunk_map.get('e-Governance Policy India', []))),
    'GDPR':                                 ('European statute', len(doc_chunk_map.get('GDPR', []))),
    'ISO 27001:2022':                       ('Intl. standard',  len(doc_chunk_map.get('ISO 27001:2022', []))),
    'NIST SP 800-66r2':                     ('Intl. standard',  len(doc_chunk_map.get('NIST SP 800-66r2', []))),
    'NIST Cybersecurity Framework':         ('Intl. standard',  len(doc_chunk_map.get('NIST Cybersecurity Framework', []))),
    'OECD Digital Security Guidelines':     ('Intl. standard',  len(doc_chunk_map.get('OECD Digital Security Guidelines', []))),
    'PCI Security Standards':               ('Intl. standard',  len(doc_chunk_map.get('PCI Security Standards', []))),
    'Privacy, Security and Ownership of Data in the Telecom Sector Takshashila Policy Advisory': ('Indian direction', len(doc_chunk_map.get('Privacy, Security and Ownership of Data in the Telecom Sector Takshashila Policy Advisory', []))),
    'NCIIPC Guidelines for Govt Entities':  ('Indian guideline', len(doc_chunk_map.get('NCIIPC Guidelines for Govt Entities', []))),
}

THRESHOLD = 0.87
net = Network(height='750px', width='100%', bgcolor='#1a1a2e', font_color='white', cdn_resources='in_line', notebook=True)

for doc, (jtype, chunks) in doc_meta.items():
    color = jurisdiction_colors.get(jtype, '#888888')
    net.add_node(doc, color=color, size=max(10, chunks * 0.8),
                 title=f'{doc}\nType: {jtype}\nChunks: {chunks}', label=doc[:22])

for sim, d1, d2 in pairs:
    if sim < THRESHOLD:
        break
    if d1 not in doc_meta or d2 not in doc_meta:
        continue
    color = '#FF0000' if sim >= 0.95 else '#FF6B00' if sim >= 0.90 else '#FFD700'
    width = 5 if sim >= 0.95 else 3 if sim >= 0.90 else 1.5
    net.add_edge(d1, d2, color=color, width=width, title=f'Similarity: {sim:.4f}')

net.set_options('{"physics": {"forceAtlas2Based": {"gravitationalConstant": -60, "springLength": 200}, "solver": "forceAtlas2Based"}}')
net.show('/kaggle/working/kg_documents.html')
print('Figure 2 saved: kg_documents.html')

In [ ]:
# Figure 3: Answer Quality Knowledge Graph
def f1_to_color(f1):
    if f1 >= 0.6:  return '#00C853'
    if f1 >= 0.5:  return '#64DD17'
    if f1 >= 0.4:  return '#FFD600'
    if f1 >= 0.3:  return '#FF6D00'
    if f1 >= 0.2:  return '#DD2C00'
    return '#B71C1C'

def jtype_shape(jtype):
    if 'Indian statute' in jtype: return 'square'
    if 'Indian' in jtype:         return 'triangle'
    if 'European' in jtype:       return 'star'
    return 'dot'

net2 = Network(height='750px', width='100%', bgcolor='#0a0a1a', font_color='white', cdn_resources='in_line', notebook=True)

doc_stats = {}
for doc, (jtype, chunks) in doc_meta.items():
    f1_rows = log_df[log_df['doc_source'].str.contains(doc[:20], na=False, regex=False)]
    mean_f1  = f1_rows['f1'].mean() if len(f1_rows) > 0 else 0.0
    n_q      = len(f1_rows)
    doc_stats[doc] = {'jtype': jtype, 'chunks': chunks, 'mean_f1': mean_f1, 'n_questions': n_q}
    if n_q > 0:
        net2.add_node(doc, label=doc[:20],
                      color=f1_to_color(mean_f1), size=max(15, n_q * 0.5),
                      shape=jtype_shape(jtype),
                      title=f'{doc}\nF1: {mean_f1:.3f}\nQuestions: {n_q}')

for sim, d1, d2 in pairs:
    if sim < 0.87:
        break
    if d1 in doc_stats and d2 in doc_stats and doc_stats[d1]['n_questions'] > 0 and doc_stats[d2]['n_questions'] > 0:
        color = '#FF1744' if sim >= 0.95 else '#FF6D00' if sim >= 0.90 else '#FFD600'
        net2.add_edge(d1, d2, color=color, width=3 if sim >= 0.90 else 1.5,
                      title=f'Similarity: {sim:.4f}')

net2.set_options('{"physics": {"forceAtlas2Based": {"gravitationalConstant": -120, "springLength": 250}, "solver": "forceAtlas2Based"}}')
net2.show('/kaggle/working/kg_answers.html')
print('Figure 3 saved: kg_answers.html')
print('\nBoth figures ready. Download and screenshot for paper.')